<a href="https://colab.research.google.com/github/Andrelhu/Simulated-PT-Patient/blob/main/claude_ver/Ana_Agent_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ana Agent — Colab Notebook

Simulated PT patient. Two backend options:
- **Ollama** (local, free) — runs a quantized model inside the Colab VM
- **Claude API** (Anthropic) — requires an API key

Run the cells top to bottom. Skip sections you don't need.

## 1 — Install Python dependencies

In [1]:
!pip install -q anthropic ollama tenacity python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 21.9 MB/s eta 0:00:00


## 2 — Clone the repo & set paths

If you already uploaded the repo to Drive, mount it and adjust `REPO_ROOT` instead.

In [2]:
import os, sys, shutil, pathlib

# ── Option A: clone from GitHub ──────────────────────────────────────────────
REPO_URL = "https://github.com/Andrelhu/Simulated-PT-Patient.git"
REPO_DIR = "/content/pt-agent"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

# ── Option B: Google Drive ────────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# REPO_DIR = "/content/drive/MyDrive/pt-agent"  # adjust path

CLAUDE_VER = os.path.join(REPO_DIR, "claude_ver")
sys.path.insert(0, CLAUDE_VER)

_pkg = os.path.join(CLAUDE_VER, "ana_agent", "__init__.py")
assert os.path.exists(_pkg), (
    f"ana_agent not found at {_pkg}.\n"
    f"Check that the repo cloned correctly: {REPO_DIR}"
)

# ── Copy spec file from Documentation/ into data/specs/ ──────────────────────
_spec_src = pathlib.Path(REPO_DIR) / "Documentation" / "ana_lopez_combined_ai_simulator_script.txt"
_specs_dir = pathlib.Path(REPO_DIR) / "data" / "specs"
_specs_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(_spec_src, _specs_dir / _spec_src.name)
print(f"Spec file ready: {_specs_dir / _spec_src.name}")

print(f"Repo:       {REPO_DIR}")
print(f"claude_ver: {CLAUDE_VER}")
print(f"ana_agent:  OK")

Cloning into '/content/pt-agent'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 36 (delta 3), reused 25 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 96.27 KiB | 5.66 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Spec file ready: /content/pt-agent/data/specs/ana_lopez_combined_ai_simulator_script.txt
Repo:       /content/pt-agent
claude_ver: /content/pt-agent/claude_ver
ana_agent:  OK


## 3 — (Ollama only) Install Ollama & pull a model

Skip this section entirely if you are using the Claude backend.

> **Recommended models for Colab free tier (T4, ~15 GB RAM)**
> - `dolphin-llama3:8b-v2.9-q4_K_M` — character-tuned, best for this simulation (~5 GB)
> - `llama3.1:8b-instruct-q4_K_M` — strong general baseline (~5 GB)
> - `llama3.2:3b-instruct-q4_K_M` — fast, ~2 GB, good for quick tests

In [3]:
# lshw + pciutils (lspci) let the Ollama installer detect the T4 GPU
# and automatically install the matching CUDA libraries.
# zstd is required to unpack the Ollama tar.zst archive.
!apt-get install -y zstd lshw pciutils 2>/dev/null | tail -1
!curl -fsSL https://ollama.com/install.sh | sh


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import os, subprocess, threading, time, urllib.request

# Helps Ollama detect the T4 GPU in Colab
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia"

def _start_ollama():
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

threading.Thread(target=_start_ollama, daemon=True).start()

# Poll until the API responds (up to 30 s) instead of a blind sleep
for _i in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1)
        print(f"Ollama server ready ({_i + 1}s)")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(
        "Ollama server didn't start in 30 s. "
        "Re-run the install cell above, then retry."
    )

Ollama server ready (2s)


In [5]:
# Change the model tag if you want a different one
OLLAMA_MODEL = "dolphin-llama3:8b-v2.9-q4_K_M"
!ollama pull {OLLAMA_MODEL}

## 4 — Configuration

Set your backend here. Only one section needs to be filled.

In [6]:
# ── Choose backend ────────────────────────────────────────────────────────────
BACKEND = "ollama"          # "ollama" | "claude"

# Ollama settings (only used when BACKEND = "ollama")
OLLAMA_MODEL      = "dolphin-llama3:8b-v2.9-q4_K_M"
OLLAMA_BASE_URL   = "http://localhost:11434"

# Claude settings (only used when BACKEND = "claude")
# Paste your key here or set the ANTHROPIC_API_KEY env var before running
ANTHROPIC_API_KEY = ""      # e.g. "sk-ant-..."
CLAUDE_MODEL      = "claude-3-5-haiku-20241022"

# Pipeline settings
GUARDRAIL         = True    # False = single-pass, faster
TEMPERATURE       = 0.7
REFRESH_TURNS     = 3       # inject constraint reminder every N turns

# ── Spec file ─────────────────────────────────────────────────────────────────
import os
DATA_DIR  = os.path.join(REPO_DIR, "data")
SPEC_FILE = os.path.join(DATA_DIR, "specs", "ana_lopez_combined_ai_simulator_script.txt")
LOGS_DIR  = os.path.join(DATA_DIR, "session_logs")

os.makedirs(LOGS_DIR, exist_ok=True)
print(f"Backend:    {BACKEND}")
print(f"Model:      {OLLAMA_MODEL if BACKEND == 'ollama' else CLAUDE_MODEL}")
print(f"Guardrail:  {GUARDRAIL}")
print(f"Spec file:  {SPEC_FILE}  exists={os.path.exists(SPEC_FILE)}")

Backend:    ollama
Model:      dolphin-llama3:8b-v2.9-q4_K_M
Guardrail:  True
Spec file:  /content/pt-agent/data/specs/ana_lopez_combined_ai_simulator_script.txt  exists=True


## 5 — Verify spec file

Confirms that the spec file was copied from `Documentation/` into `data/specs/`.
If it's missing, re-run the clone cell above.

In [7]:
from pathlib import Path

if not Path(SPEC_FILE).exists():
    raise FileNotFoundError(
        f"Spec file not found: {SPEC_FILE}\n"
        "Re-run the clone cell above to copy it from Documentation/."
    )
print(f"Spec file OK: {Path(SPEC_FILE).name} ({Path(SPEC_FILE).stat().st_size} bytes)")

Spec file OK: ana_lopez_combined_ai_simulator_script.txt (9989 bytes)


## 6 — Initialise provider and session

In [8]:
import sys, os
# Guard: re-add path if kernel was restarted after cell 2
_claude_ver = "/content/pt-agent/claude_ver"
if _claude_ver not in sys.path:
    sys.path.insert(0, _claude_ver)

from pathlib import Path
from ana_agent.providers import build_provider
from ana_agent.session import ConversationHistory

if BACKEND == "claude" and ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

MODEL = OLLAMA_MODEL if BACKEND == "ollama" else CLAUDE_MODEL

provider = build_provider(
    backend=BACKEND,
    model=MODEL,
    anthropic_api_key=os.environ.get("ANTHROPIC_API_KEY", ""),
    ollama_base_url=OLLAMA_BASE_URL,
)

history = ConversationHistory()
print(f"Provider ready: {BACKEND} / {MODEL}")

Provider ready: ollama / dolphin-llama3:8b-v2.9-q4_K_M


## 7 — Chat helper

Run this cell once to define the `chat()` function.

In [9]:
import datetime as dt
from pathlib import Path
from ana_agent.pipeline import run_pipeline

_log_path = Path(LOGS_DIR) / f"colab_{BACKEND}_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

def chat(message: str) -> str:
    """Send a message to Ana and print her reply."""
    response = run_pipeline(
        user_input=message,
        history=history,
        provider=provider,
        spec_file=Path(SPEC_FILE),
        temperature=TEMPERATURE,
        guardrail=False,
        refresh_turns=REFRESH_TURNS,
    )
    history.append_user(message)
    history.append_assistant(response)

    with _log_path.open("a", encoding="utf-8") as f:
        f.write(f"USER: {message}\nANA:  {response}\n\n")

    return response


def reset_session():
    """Clear conversation history and start a new log file."""
    global history, _log_path
    history = ConversationHistory()
    _log_path = Path(LOGS_DIR) / f"colab_{BACKEND}_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    print("Session reset.")


print("chat() and reset_session() ready.")
print(f"Log: {_log_path}")

chat() and reset_session() ready.
Log: /content/pt-agent/data/session_logs/colab_ollama_20260423_140124.txt


## 8 — Voice Input (Whisper STT)

Records from the browser microphone and transcribes to text using
`faster-whisper tiny` on CPU — GPU stays fully dedicated to Ollama.

In [10]:
!pip install -q faster-whisper soundfile
!apt-get install -y ffmpeg 2>/dev/null | tail -1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 90.4 MB/s eta 0:00:00
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [11]:
from faster_whisper import WhisperModel

STT_MODEL = WhisperModel("tiny", device="cpu", compute_type="int8")
print("Whisper tiny loaded on CPU.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Whisper tiny loaded on CPU.


In [12]:
from google.colab.output import eval_js
from IPython.display import display
import base64, subprocess

_RECORD_JS = """
async function record(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream, {mimeType: 'audio/webm'});
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.start();
  await new Promise(r => setTimeout(r, ms));
  rec.stop();
  await new Promise(r => rec.onstop = r);
  stream.getTracks().forEach(t => t.stop());
  const blob = new Blob(chunks);
  const reader = new FileReader();
  reader.readAsDataURL(blob);
  await new Promise(r => reader.onload = r);
  return reader.result;
}
"""

def record_audio(duration_s=5):
    """Record from browser mic; return path to WAV file."""
    print(f"Recording {duration_s}s — speak now...")
    data = eval_js(_RECORD_JS + f"record({duration_s * 1000})")
    _, encoded = data.split(',', 1)
    audio_bytes = base64.b64decode(encoded)
    webm, wav = '/tmp/student.webm', '/tmp/student.wav'
    with open(webm, 'wb') as f:
        f.write(audio_bytes)
    subprocess.run(
        ['ffmpeg', '-y', '-i', webm, '-ar', '16000', '-ac', '1', wav],
        capture_output=True
    )
    return wav

def transcribe(wav_path):
    """Transcribe WAV to text using Whisper tiny (CPU)."""
    segments, _ = STT_MODEL.transcribe(wav_path, beam_size=1)
    return " ".join(seg.text for seg in segments).strip()

print("record_audio() and transcribe() ready.")

record_audio() and transcribe() ready.


## 9 — Voice Output (Piper TTS)

Converts Ana's text responses to speech using Piper (CPU).
Voice: `en_US-amy-medium` — female, American English.

In [13]:
import pathlib

PIPER_DIR   = pathlib.Path("/content/piper")
VOICE_DIR   = pathlib.Path("/content/piper_voices")
VOICE_MODEL = str(VOICE_DIR / "en_US-amy-medium.onnx")
VOICE_CFG   = VOICE_MODEL + ".json"
_BASE_VOICE = "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/amy/medium"

if not PIPER_DIR.exists():
    !wget -q https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_linux_x86_64.tar.gz -O /tmp/piper.tar.gz
    !tar -xzf /tmp/piper.tar.gz -C /content/
    print("Piper installed.")
else:
    print("Piper already present.")

VOICE_DIR.mkdir(exist_ok=True)
if not pathlib.Path(VOICE_MODEL).exists():
    !wget -q {_BASE_VOICE}/en_US-amy-medium.onnx      -O {VOICE_MODEL}
    !wget -q {_BASE_VOICE}/en_US-amy-medium.onnx.json -O {VOICE_CFG}
    print("Voice model downloaded.")
else:
    print("Voice model already present.")


gzip: stdin: unexpected end of file
tar: Child returned status 1
tar: Error is not recoverable: exiting now
Piper installed.
Voice model downloaded.


In [14]:
from IPython.display import Audio

def speak(text):
    """Convert text to speech with Piper and play inline."""
    out = '/tmp/ana_response.wav'
    proc = subprocess.run(
        [str(PIPER_DIR / "piper"),
         "--model", VOICE_MODEL,
         "--output_file", out],
        input=text.encode(),
        capture_output=True,
    )
    if proc.returncode != 0:
        print(f"[TTS error: {proc.stderr.decode()}]")
        return
    return Audio(out, autoplay=True)

print("speak() ready.")

speak() ready.


## 10 — Voice Chat

Records the student's question, sends it to Ana, and plays her response aloud.
Default recording is 5 s — increase for longer questions.

In [15]:
def voice_chat(duration_s=5):
    """Full voice loop: mic → Whisper STT → Ana → Piper TTS."""
    wav       = record_audio(duration_s)
    user_text = transcribe(wav)
    if not user_text:
        print("No speech detected — try again.")
        return
    print(f"you> {user_text}")
    response  = chat(user_text)
    print(f"ana> {response}")
    return speak(response)

print("voice_chat() ready.")
print("Usage: voice_chat(5)  — record 5 s, transcribe, get Ana's voice reply.")

voice_chat() ready.
Usage: voice_chat(5)  — record 5 s, transcribe, get Ana's voice reply.


In [17]:
voice_chat()

Recording 5s — speak now...
No speech detected — try again.


## 11 — Talk to Ana (text)

Call `chat("your message")` in any cell below.
Each call appends to the running conversation history.

In [ ]:
chat("Can you tell me what brings you in today?")

In [ ]:
chat("Did you get referred by a doctor, or have you seen any other healthcare providers about this injury?")

In [ ]:
chat("Where is your pain located?")

Can you describe the pain? Is it sharp, dull, achy, stabbing, something else?
What makes your pain better or worse?
Can you describe how the injury happened?
Have you ever had an injury like this before?
Do you have any numbness or tingling?
What kinds of things have been difficult since you started having this pain?
What do you want to get back to doing that you can’t do right now?

## 12 — Interactive loop (optional)

Type messages one at a time. Type `exit` to stop.

In [ ]:
while True:
    user = input("you> ").strip()
    if not user:
        continue
    if user.lower() in {"exit", "quit", "/exit"}:
        print("Session ended.")
        break
    chat(user)
    print()

## 13 — Review session log

In [ ]:
print(_log_path.read_text(encoding="utf-8"))

## 14 — Reset and start a new session

In [ ]:
reset_session()